# 🎙️ ElevenLabs Russian Studio (XTTS-v2 Engine)
### Идеальное клонирование голоса на **русском языке** без акцента
---
> * 🇷🇺 **Нативный русский язык**: Обучена на десятках тысяч часов русской речи.
> * 🎯 **Точный тембр**: Копирует интонацию, басы, высоту голоса и эмоции.
> * 📱 **WebUI**: Работает прямо в браузере телефона и ПК через защищенную ссылку.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markankamaransheogarat-sys/open-elevenlabs-voice-studio/blob/main/Open_ElevenLabs_Voice_Studio_F5_TTS.ipynb)

## 1. ⚙️ Установка зависимостей
Нажмите кнопку **Play** на блоке ниже. Установка займет около 45 секунд.

In [ ]:
# @title ⚙️ 1. Установка окружения (Run Setup)
import os
os.environ['COQUI_TOS_AGREED'] = '1'

!nvidia-smi
print('📦 Установка движка клонирования XTTS-v2 с нативным русским языком...')
!pip install -q coqui-tts soundfile gradio
!apt-get install -y -qq ffmpeg
print('✅ Окружение успешно настроено!')


## 2. 🚀 Запуск веб-студии
Запустите блок ниже. Через ~1 минуту появится ссылка **`Running on public URL: https://xxxx.gradio.live`**.
Перейдите по ней в браузере.

In [ ]:
# @title 🚀 2. Запуск ElevenLabs Russian Studio (Launch WebUI)
import os
import torch
import gradio as gr
import tempfile
from TTS.api import TTS

os.environ['COQUI_TOS_AGREED'] = '1'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'
print(f'⚡ Устройство: {device} ({dev_name})')

print('📥 Загрузка нейросети XTTS-v2 (нативная поддержка русского языка)...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('🔥 Модель готова к генерации чистого русского языка!')

def clone_voice(ref_audio, text, language, speed):
    if not ref_audio:
        raise gr.Error('⚠️ Пожалуйста, загрузите или запишите образец голоса (5–15 секунд)!')
    if not text or not text.strip():
        raise gr.Error('⚠️ Введите текст для озвучки!')
    
    out_file = tempfile.mktemp(suffix='.wav')
    tts.tts_to_file(
        text=text.strip(),
        speaker_wav=ref_audio,
        language=language,
        speed=float(speed),
        file_path=out_file
    )
    return out_file

custom_css = '''
.gradio-container { font-family: 'Inter', sans-serif; max-width: 1000px !important; margin: auto; }
#btn-synth { background: linear-gradient(135deg, #f97316 0%, #ea580c 100%); color: white; font-weight: bold; font-size: 16px; border-radius: 8px; }
'''

with gr.Blocks(title='ElevenLabs Russian Studio | XTTS-v2', css=custom_css, theme=gr.themes.Soft(primary_hue='orange', neutral_hue='slate')) as demo:
    gr.Markdown(
        '''
        # 🎙️ ElevenLabs Russian Studio (XTTS-v2 Engine)
        ### Нативное клонирование любого голоса на **русском языке**
        ---
        '''
    )
    with gr.Row():
        with gr.Column(scale=1):
            ref_audio = gr.Audio(
                label='1️⃣ Образец голоса (Reference Audio, 5-15 сек)',
                type='filepath',
                sources=['upload', 'microphone']
            )
            lang = gr.Dropdown(
                label='Язык озвучки (Language)',
                choices=[('Русский (Russian)', 'ru'), ('English', 'en'), ('Deutsch', 'de'), ('Español', 'es'), ('Français', 'fr')],
                value='ru'
            )
            speed = gr.Slider(label='Скорость речи (Speed)', minimum=0.7, maximum=1.5, value=1.0, step=0.05)
            
        with gr.Column(scale=1):
            text_input = gr.Textbox(
                label='2️⃣ Текст для озвучки клонированным голосом',
                placeholder='Привет! Теперь я говорю на чистом русском языке с твоим голосом, интонацией и тембром.',
                lines=6
            )
            btn_synth = gr.Button('⚡ Сгенерировать речь клонированным голосом', elem_id='btn-synth', size='lg')
            out_audio = gr.Audio(label='3️⃣ Готовая озвучка (WAV)', type='filepath')
            
            btn_synth.click(
                fn=clone_voice,
                inputs=[ref_audio, text_input, lang, speed],
                outputs=[out_audio]
            )

print('👇 Скопируйте ссылку gradio.live ниже:')
demo.queue().launch(share=True, debug=False)


## 3. 🌐 API для Telegram-ботов и скриптов

In [ ]:
# @title 🌐 3. Вызов через Python API (Telegram-боты, скрипты)
'''
from gradio_client import Client, handle_file

# Вставьте полученную ссылку xxxx.gradio.live
client = Client('https://YOUR-LINK.gradio.live/')
result = client.predict(
    ref_audio=handle_file('my_sample.wav'),
    text='Привет из Telegram-бота!',
    language='ru',
    speed=1.0,
    api_name='/clone_voice'
)
print('Готовое аудио:', result)
'''
